<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.10-human-in-the-loop/notebooks/exercises-9_10.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.10: Human-in-the-Loop Patterns

*Module 9 · AI Agents & Autonomous Systems*

> Fully autonomous agents are powerful but dangerous. The most reliable production agents include human checkpoints : approval gates before destructive actions, confidence thresholds that escalate uncertain decisions, and graceful handoffs to human operators. This lesson codes four patterns that make agents safe for real deployment.

`Approval Gates` · `Confidence Thresholds` · `Escalation` · `Gradual Autonomy` · `60 min`

---

## About this notebook

This notebook contains the **2 runnable code examples** from the published lesson `lesson-9.10.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Approval Gates — Pause Before Dangerous Actions — `01_approval_gate.py`
2. Step 2 — Confidence Thresholds — Auto-Execute When Certain — `02_confidence.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Approval Gates — Pause Before Dangerous Actions

Agent proposes an action. Human approves or rejects. Only then execute.

**`01_approval_gate.py`** · _Block 1 of 2_

APPROVAL GATE — Pause for human approval before destructive actions


In [ ]:
# APPROVAL GATE — Pause for human approval before destructive actions
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ApprovalGateAgent:
    """Agent that pauses for approval on high-risk actions."""
    REQUIRES_APPROVAL = ["send_email", "delete", "purchase", "book_course", "process_refund"]

    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.pending_approval = None

    def _decide_action(self, task):
        resp = self.model.generate_content(
            f"Task: {task}\nTools: search, calculate, send_email, book_course, process_refund\n"
            f'Return JSON: {{"action":"tool_name","args":{{"key":"value"}},"reason":"why"}}')
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return {"action":"search","args":{},"reason":raw[:100]}

    def _needs_approval(self, action):
        return action.get("action","") in self.REQUIRES_APPROVAL

    def run(self, task, auto_approve=False):
        action = self._decide_action(task)
        print(f"  Action: {action['action']}({action.get('args',{})})")
        print(f"  Reason: {action.get('reason','')[:60]}")

        if self._needs_approval(action):
            print(f"  ⚠  REQUIRES APPROVAL: {action['action']}")
            if auto_approve:
                print(f"  ✓ Auto-approved (demo mode)")
            else:
                self.pending_approval = action
                return {"status":"waiting_approval", "action":action}
        else:
            print(f"  ✓ Safe action, executing automatically")

        return {"status":"executed", "action":action}

    def approve(self):
        if self.pending_approval:
            action = self.pending_approval
            self.pending_approval = None
            return {"status":"approved_and_executed", "action":action}

    def reject(self, reason=""):
        self.pending_approval = None
        return {"status":"rejected", "reason":reason}

# Demo
agent = ApprovalGateAgent()
print("Approval Gate Agent:\n")
for task in ["Search for the GenAI course price",         # safe: auto-execute
            "Book the GenAI course for [email protected]",  # dangerous: needs approval
            "Process a refund for order #12345"]:            # dangerous: needs approval
    print(f"  Task: {task}")
    r = agent.run(task, auto_approve=True)
    print(f"  Status: {r['status']}\n")


> **What just happened?** “Search for price” → safe, auto-executed. “Book course” → book_course is in REQUIRES_APPROVAL, agent pauses. “Process refund” → same. The approval list is the safety boundary. Read-only actions execute freely. Actions that spend money, send emails, or delete data require human sign-off.


### Step 2 · Confidence Thresholds — Auto-Execute When Certain

Agent rates its own confidence. High = auto-execute. Low = escalate to human.

**`02_confidence.py`** · _Block 2 of 2_

CONFIDENCE THRESHOLD — Auto-execute vs escalate


In [ ]:
# CONFIDENCE THRESHOLD — Auto-execute vs escalate
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ConfidenceAgent:
    """Auto-act when confident, escalate when uncertain."""
    def __init__(self, threshold=0.8):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.threshold = threshold
        self.escalation_queue = []

    def respond(self, query, context=""):
        resp = self.model.generate_content(
            f"Answer this query from context. Also rate your confidence 0.0-1.0.\n"
            f"Context: {context}\nQuery: {query}\n"
            f'Return JSON: {{"answer":"...","confidence":0.0-1.0,"reasoning":"why this confidence"}}')
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: result = json.loads(raw)
        except: result = {"answer":raw[:200],"confidence":0.5}

        conf = result.get("confidence", 0.5)
        if conf >= self.threshold:
            return {"answer":result["answer"], "confidence":conf,
                    "action":"auto_responded", "escalated":False}
        else:
            self.escalation_queue.append({"query":query, "draft":result["answer"],
                "confidence":conf, "reasoning":result.get("reasoning","")})
            return {"answer":"I'm not fully confident. Let me check with a specialist.",
                    "confidence":conf, "action":"escalated", "escalated":True,
                    "draft":result["answer"]}

# Demo
kb = "Netsetos GenAI: 14999 INR, 146hrs, refund 7 days full, 8-30 days 50%."
agent = ConfidenceAgent(threshold=0.8)

print("Confidence Threshold Agent:\n")
for q in ["How much does the GenAI course cost?",          # high confidence
          "Can I use the certificate for a US visa application?",  # low confidence
          "What is the refund policy?",                      # high confidence
          "Will GenAI replace my data science job?"]:          # low confidence
    r = agent.respond(q, kb)
    icon = "✓" if not r["escalated"] else "⚠"
    print(f"  {icon} [{r['confidence']:.1f}] {q[:45]}")
    print(f"    → {r['action']}: {r['answer'][:70]}\n")

print(f"  Escalation queue: {len(agent.escalation_queue)} items")


> **What just happened?** Escalation is not just about routing — it is about routing with CONTEXT. Pass conversation history, classification scores, and suggested next steps so reviewers start informed, not blind.


---

## Wrap-up

You've walked through all 2 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
